# XGBoost vs CatBoost — Promoter Expression Prediction
Improved XGBoost + CatBoost comparison on 6 gene-expression conditions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
import warnings
from itertools import product as iproduct

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

XLSX_PATH = '/content/capstone_dataset.xlsx'

print('Reading Excel file ...')
wb = openpyxl.load_workbook(XLSX_PATH, read_only=True, data_only=True)

ws1 = wb['Supplementary Table 1']
rows1 = list(ws1.iter_rows(values_only=True))[4:]
seq_dict = {r[0]: r[11] for r in rows1 if r[0] and r[11]}
print(f'Sequences loaded: {len(seq_dict):,}')

ws2 = wb['Supplementary Table 2']
rows2 = list(ws2.iter_rows(values_only=True))[4:]
exp_dict = {r[0]: {'species': r[1], 'expr': r[2:8]} for r in rows2 if r[0]}
print(f'Expression records: {len(exp_dict):,}')

COND_COLS = [
    'no_enhancer_dark_tobacco',
    'no_enhancer_dark_maize',
    'with_enhancer_dark_tobacco',
    'with_enhancer_dark_maize',
    'no_enhancer_light_tobacco',
    'with_enhancer_light_tobacco',
]

records = []
for gene in set(seq_dict) & set(exp_dict):
    vals = exp_dict[gene]['expr']
    if all(v is not None and v != '#N/A' for v in vals):
        row = {'gene': gene, 'species': exp_dict[gene]['species'], 'sequence': seq_dict[gene]}
        for col, v in zip(COND_COLS, vals):
            row[col] = float(v)
        records.append(row)

df = pd.DataFrame(records)
print(f'\nMerged dataset: {len(df):,} promoters')

## Preprocessing & Feature Engineering

In [ ]:
df = df.dropna(subset=['sequence']).copy()
df['sequence'] = df['sequence'].str.upper()
valid_chars = set("ATCG")
df = df[df['sequence'].apply(lambda x: set(x).issubset(valid_chars))].copy()
print("After cleaning:", df.shape)

In [ ]:
SEQ_LEN = 170

def fix_sequence_length(seq, length=SEQ_LEN):
    seq = seq[:length]
    if len(seq) < length:
        seq = seq + 'A' * (length - len(seq))
    return seq

df['sequence'] = df['sequence'].apply(lambda x: fix_sequence_length(x, SEQ_LEN))

# ── NEW: log-transform targets to improve R² ────────────────────────────────
# Shift to ensure all values > 0 before log, then inverse-transform for metrics
for col in COND_COLS:
    df[col] = np.log1p(df[col] - df[col].min() + 1e-6)

print("Sequences standardised. Targets log-transformed.")
df[COND_COLS].describe().round(3)

In [ ]:
def one_hot_encode(seq, seq_len=SEQ_LEN):
    mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
    mat = np.zeros((4, seq_len), dtype=np.float32)
    for i, nt in enumerate(seq[:seq_len]):
        if nt in mapping:
            mat[mapping[nt], i] = 1.0
    return mat.flatten()

def kmer_features(seq, k=4):
    kmers = [''.join(p) for p in iproduct('ATCG', repeat=k)]
    kmer_idx = {km: i for i, km in enumerate(kmers)}
    vec = np.zeros(len(kmers), dtype=np.float32)
    for i in range(len(seq) - k + 1):
        km = seq[i:i+k]
        if km in kmer_idx:
            vec[kmer_idx[km]] += 1
    total = vec.sum()
    if total > 0:
        vec /= total
    return vec

def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)

def gc_windows(seq, window_size=20):
    vals = []
    for i in range(0, len(seq), window_size):
        window = seq[i:i+window_size]
        if window:
            gc = (window.count('G') + window.count('C')) / len(window)
            vals.append(gc)
    return np.array(vals, dtype=np.float32)

MOTIFS = ['TATA', 'CAAT', 'CGCG', 'GATA', 'AATAAA', 'CCAAT', 'ATTGG', 'GCGC']  # extended motif list

def motif_features(seq, motifs=MOTIFS):
    feats = []
    for m in motifs:
        count = seq.count(m)
        # Also capture position of first occurrence
        pos = seq.find(m)
        feats.extend([count, pos / SEQ_LEN if pos >= 0 else -1.0])
    return np.array(feats, dtype=np.float32)

# NEW: dinucleotide composition
def dinuc_features(seq):
    bases = 'ATCG'
    di = [''.join(p) for p in iproduct(bases, bases)]
    di_idx = {d: i for i, d in enumerate(di)}
    vec = np.zeros(len(di), dtype=np.float32)
    for i in range(len(seq) - 1):
        d = seq[i:i+2]
        if d in di_idx:
            vec[di_idx[d]] += 1
    total = vec.sum()
    if total > 0:
        vec /= total
    return vec

print("Feature functions defined (with dinucleotides + extended motifs).")

In [ ]:
species_dummies = pd.get_dummies(df['species'], prefix='species')

X_features = []
for i, row in df.iterrows():
    seq = row['sequence']
    onehot  = one_hot_encode(seq)
    kmer4   = kmer_features(seq, k=4)
    kmer3   = kmer_features(seq, k=3)   # NEW: tri-mer features
    gc      = np.array([gc_content(seq)], dtype=np.float32)
    gc_win  = gc_windows(seq, window_size=20)
    motifs  = motif_features(seq)
    dinuc   = dinuc_features(seq)       # NEW
    sp_feat = species_dummies.loc[i].values.astype(np.float32)
    combined = np.concatenate([onehot, kmer4, kmer3, gc, gc_win, motifs, dinuc, sp_feat])
    X_features.append(combined)

X = np.array(X_features, dtype=np.float32)
print("Final X shape:", X.shape)

In [ ]:
Y = df[COND_COLS].values.astype(np.float32)
indices = np.arange(len(df))
train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=42)
val_idx,  test_idx  = train_test_split(temp_idx, test_size=0.50, random_state=42)

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
Y_train, Y_val, Y_test = Y[train_idx], Y[val_idx], Y[test_idx]

print("Train:", X_train.shape, Y_train.shape)
print("Val  :", X_val.shape,   Y_val.shape)
print("Test :", X_test.shape,  Y_test.shape)

In [ ]:
def print_metrics(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f"{name:12s}  R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")

## Improved XGBoost Training

In [ ]:
xgb_models = {}
xgb_results = []

for j, col in enumerate(COND_COLS):
    print(f"\n{'='*50}")
    print(f"XGBoost → {col}")
    y_train = Y_train[:, j]
    y_val   = Y_val[:,   j]
    y_test  = Y_test[:,  j]

    model = XGBRegressor(
        n_estimators     = 1500,      # more trees
        max_depth        = 7,         # slightly deeper
        learning_rate    = 0.02,      # lower LR → better generalisation
        subsample        = 0.85,
        colsample_bytree = 0.75,
        colsample_bylevel= 0.75,      # NEW: column sampling per level
        reg_alpha        = 0.1,       # L1 regularisation
        reg_lambda       = 1.5,       # L2 regularisation
        min_child_weight = 5,         # prevent overfitting
        gamma            = 0.1,       # minimum gain to split
        objective        = 'reg:squarederror',
        tree_method      = 'hist',    # faster for large datasets
        random_state     = 42,
        n_jobs           = -1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    train_pred = model.predict(X_train)
    val_pred   = model.predict(X_val)
    test_pred  = model.predict(X_test)

    print_metrics("Train",      y_train, train_pred)
    print_metrics("Validation", y_val,   val_pred)
    print_metrics("Test",       y_test,  test_pred)

    xgb_models[col] = model
    xgb_results.append({
        'condition': col,
        'model': 'XGBoost',
        'train_r2': r2_score(y_train, train_pred),
        'val_r2':   r2_score(y_val,   val_pred),
        'test_r2':  r2_score(y_test,  test_pred),
        'test_rmse':np.sqrt(mean_squared_error(y_test, test_pred)),
        'test_mae': mean_absolute_error(y_test, test_pred),
    })

xgb_df = pd.DataFrame(xgb_results)
print("\n=== XGBoost Summary ===")
print(xgb_df[['condition','test_r2','test_rmse','test_mae']].to_string(index=False))
print(f"\nMean Test R²: {xgb_df['test_r2'].mean():.4f}")

## CatBoost Training

In [ ]:
cat_models = {}
cat_results = []

for j, col in enumerate(COND_COLS):
    print(f"\n{'='*50}")
    print(f"CatBoost → {col}")
    y_train = Y_train[:, j]
    y_val   = Y_val[:,   j]
    y_test  = Y_test[:,  j]

    model = CatBoostRegressor(
        iterations         = 1500,
        depth              = 8,
        learning_rate      = 0.02,
        l2_leaf_reg        = 3.0,
        subsample          = 0.85,
        colsample_bylevel  = 0.75,
        min_data_in_leaf   = 20,
        loss_function      = 'RMSE',
        eval_metric        = 'R2',
        random_seed        = 42,
        thread_count       = -1,
        verbose            = False,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=100,
        verbose=False,
    )

    train_pred = model.predict(X_train)
    val_pred   = model.predict(X_val)
    test_pred  = model.predict(X_test)

    print_metrics("Train",      y_train, train_pred)
    print_metrics("Validation", y_val,   val_pred)
    print_metrics("Test",       y_test,  test_pred)

    cat_models[col] = model
    cat_results.append({
        'condition': col,
        'model': 'CatBoost',
        'train_r2': r2_score(y_train, train_pred),
        'val_r2':   r2_score(y_val,   val_pred),
        'test_r2':  r2_score(y_test,  test_pred),
        'test_rmse':np.sqrt(mean_squared_error(y_test, test_pred)),
        'test_mae': mean_absolute_error(y_test, test_pred),
    })

cat_df = pd.DataFrame(cat_results)
print("\n=== CatBoost Summary ===")
print(cat_df[['condition','test_r2','test_rmse','test_mae']].to_string(index=False))
print(f"\nMean Test R²: {cat_df['test_r2'].mean():.4f}")

## Side-by-Side Comparison

In [ ]:
all_results = pd.concat([xgb_df, cat_df], ignore_index=True)

comparison = all_results.pivot_table(
    index='condition',
    columns='model',
    values=['test_r2', 'test_rmse', 'test_mae']
).round(4)

print("\n=== Test Set Comparison ===")
print(comparison.to_string())

# Summary row
print("\n=== Mean Across All Conditions ===")
summary = all_results.groupby('model')[['test_r2','test_rmse','test_mae']].mean().round(4)
print(summary.to_string())

## Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── R² bar chart ────────────────────────────────────────────────────────────
ax = axes[0]
x = np.arange(len(COND_COLS))
w = 0.35
short_labels = [c.replace('_', '\n') for c in COND_COLS]

b1 = ax.bar(x - w/2, xgb_df['test_r2'], w, label='XGBoost', color='#1a7fc1', alpha=0.85)
b2 = ax.bar(x + w/2, cat_df['test_r2'],  w, label='CatBoost', color='#e05c2d', alpha=0.85)
ax.set_xlabel('Condition', fontsize=11)
ax.set_ylabel('Test R²', fontsize=11)
ax.set_title('Test R² — XGBoost vs CatBoost', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=8)
ax.legend()
ax.set_ylim(0, 1)
ax.axhline(0.5, color='grey', linestyle='--', alpha=0.5)

# ── RMSE bar chart ───────────────────────────────────────────────────────────
ax = axes[1]
b3 = ax.bar(x - w/2, xgb_df['test_rmse'], w, label='XGBoost', color='#1a7fc1', alpha=0.85)
b4 = ax.bar(x + w/2, cat_df['test_rmse'],  w, label='CatBoost', color='#e05c2d', alpha=0.85)
ax.set_xlabel('Condition', fontsize=11)
ax.set_ylabel('Test RMSE', fontsize=11)
ax.set_title('Test RMSE — XGBoost vs CatBoost', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=8)
ax.legend()

plt.tight_layout()
plt.savefig('/home/claude/comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")

In [ ]:
# Scatter: predicted vs actual for best condition (per model)
best_cond = xgb_df.loc[xgb_df['test_r2'].idxmax(), 'condition']
j = COND_COLS.index(best_cond)

xgb_pred = xgb_models[best_cond].predict(X_test)
cat_pred  = cat_models[best_cond].predict(X_test)
y_true    = Y_test[:, j]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, name, color in zip(
    axes,
    [xgb_pred, cat_pred],
    ['XGBoost', 'CatBoost'],
    ['#1a7fc1', '#e05c2d']
):
    r2 = r2_score(y_true, pred)
    ax.scatter(y_true, pred, alpha=0.3, s=5, color=color)
    lo, hi = min(y_true.min(), pred.min()), max(y_true.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5)
    ax.set_xlabel('True Expression (log-scaled)', fontsize=11)
    ax.set_ylabel('Predicted', fontsize=11)
    ax.set_title(f'{name} | {best_cond}\nR²={r2:.4f}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/home/claude/scatter_best.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
all_results.to_csv('/content/xgb_catboost_comparison.csv', index=False)
print("Results saved to xgb_catboost_comparison.csv")
print("\nFinal Summary:")
print(all_results.groupby('model')[['test_r2','test_rmse','test_mae']].mean().round(4))